# RETINOVA — Starter Pipeline (Google Colab)
Diabetic Retinopathy Detection — APTOS 2019 dataset 

**how to exceute**
1. Runtime → Change runtime type → GPU (T4) 

3. Cell 2 Kaggle API token (`kaggle.json`) upload 

**Pipeline:** Dataset download → Preprocessing (CLAHE) → EfficientNetV2 Training → Evaluation → Grad-CAM Explainability


## Step 1: Setup — Libraries installation

In [ ]:
!pip install -q kaggle timm grad-cam albumentations opencv-python-headless
print("Libraries installed ✔")


## Step 2: Kaggle API Setup
Kaggle.com → Account → Settings → API → "Create New Token" pe click karo.
Isse `kaggle.json` file download hogi. Neeche wale cell run karke usko upload krni padegi 


In [ ]:
from google.colab import files
import os

print(" upload kaggle.json file  :")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
print("Kaggle API ready ✔")


## Step 3: Dataset Download (APTOS 2019)

https://www.kaggle.com/c/aptos2019-blindness-detection/rules


In [ ]:
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data
print("Dataset downloaded aur extract ho gaya ")


## Step 4: Imports

In [ ]:
import pandas as pd
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Step 5: Preprocessing (CLAHE + Resize)
Ye Module 2 (Image Quality Assessment & Preprocessing) ka core hai — CLAHE contrast enhance karta hai
taaki lesions (microaneurysms, hemorrhages) zyada clearly dikhein.


In [ ]:
def preprocess_image(img_path, img_size=224):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # CLAHE apply karo (sirf L channel pe, LAB color space mein)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)

    img_resized = cv2.resize(img_clahe, (img_size, img_size))
    return img_resized

# Quick visual check — before vs after CLAHE
train_df = pd.read_csv('/content/data/train.csv')
sample_path = f"/content/data/train_images/{train_df.iloc[0]['id_code']}.png"

original = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
processed = preprocess_image(sample_path)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(original); ax[0].set_title("Original"); ax[0].axis('off')
ax[1].imshow(processed); ax[1].set_title("After CLAHE + Resize"); ax[1].axis('off')
plt.show()


## Step 6: Dataset class aur Train/Val split

In [ ]:
class RetinaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.img_dir}/{row['id_code']}.png"
        img = preprocess_image(img_path)
        label = int(row['diagnosis'])

        if self.transform:
            img = self.transform(image=img)['image']

        return img, label

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_split, val_split = train_test_split(train_df, test_size=0.2, stratify=train_df['diagnosis'], random_state=42)

train_ds = RetinaDataset(train_split, '/content/data/train_images', transform=train_transform)
val_ds = RetinaDataset(val_split, '/content/data/train_images', transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")


## Step 7: Model — EfficientNetV2 (Module 4: DR Classification)
Pretrained EfficientNetV2 use kar rahe hain (ImageNet weights), sirf final layer 5 classes ke liye replace kar rahe hain.


In [ ]:
model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

print("Model ready:", sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable params")


## Step 8: Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return total_loss / len(loader), acc, kappa, all_preds, all_labels

EPOCHS = 10  # shuru mein kam rakho, baad mein badha sakte ho
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_kappa": []}

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_kappa, _, _ = validate(model, val_loader, criterion)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_kappa"].append(val_kappa)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | QWK: {val_kappa:.4f}")

torch.save(model.state_dict(), '/content/retinova_efficientnetv2.pth')
print("Model saved ✔")


## Step 9: Results — Training curves + Confusion Matrix

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], label="Train Loss")
ax[0].plot(history["val_loss"], label="Val Loss")
ax[0].set_title("Loss Curve"); ax[0].legend()

ax[1].plot(history["val_acc"], label="Val Accuracy")
ax[1].plot(history["val_kappa"], label="Quadratic Weighted Kappa")
ax[1].set_title("Validation Metrics"); ax[1].legend()
plt.show()

_, _, _, preds, labels = validate(model, val_loader, criterion)
cm = confusion_matrix(labels, preds)
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(5), class_names, rotation=45)
plt.yticks(range(5), class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
for i in range(5):
    for j in range(5):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout()
plt.show()

print(classification_report(labels, preds, target_names=class_names))


## Step 10: Explainable AI — Grad-CAM (Module 5 ka glimpse)
Model kahan dekh raha hai lesion detect karne ke liye — heatmap se pata chalega.


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

target_layer = [model.conv_head]  # EfficientNetV2 ka last conv layer
cam = GradCAM(model=model, target_layers=target_layer)

# ek val sample lo
sample_img, sample_label = val_ds[0]
input_tensor = sample_img.unsqueeze(0).to(device)

grayscale_cam = cam(input_tensor=input_tensor)[0]

# original (unnormalized) image visualization ke liye
img_for_display = preprocess_image(f"/content/data/train_images/{val_split.iloc[0]['id_code']}.png") / 255.0
visualization = show_cam_on_image(img_for_display, grayscale_cam, use_rgb=True)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img_for_display); ax[0].set_title(f"Original (Label: {class_names[sample_label]})"); ax[0].axis('off')
ax[1].imshow(visualization); ax[1].set_title("Grad-CAM Heatmap"); ax[1].axis('off')
plt.show()


---
## Agla kadam (next steps)
- Isi structure mein Swin Transformer aur RETFound add karke 3-way comparison table banao
- IDRiD dataset pe UNet++ train karo (`segmentation-models-pytorch` library use karo) — lesion segmentation ke liye
- Confidence threshold laga ke Refer/Monitor/Clear logic likho
- Streamlit se simple demo UI bana lo
